# Student 3 — Defect Detection
## Notebook 03: RT-DETR (real-time detection transformer)

**Algorithm.** RT-DETR is the Transformer-based end-to-end detector of Section 2.4.3. Two design
choices make a DETR fast enough for real time: the **efficient hybrid encoder**, which applies
intra-scale self-attention (AIFI) only to the smallest, highest-level feature map `S5` and fuses the
larger maps `S3`/`S4` with cheap convolutions (CCFM); and **uncertainty-minimal query selection**,
which seeds the decoder with queries that score well on classification *and* localisation instead of
classification alone. Like YOLOv10 it is NMS-free, so its latency does not grow with the number of
defects on the board — the property that matters on densely populated PCBs.

**Prerequisite.** Run `00_data_preparation.ipynb` first. This notebook consumes
`dataset/yolo/data.yaml` and writes `results/rtdetr.json`.

### 1. Environment

RT-DETR ships inside the same Ultralytics package used for YOLOv10, so no additional install is
needed if notebook 01 has already been run. RT-DETR is markedly heavier than YOLOv10-nano: a GPU is
strongly recommended, and the batch size below is reduced accordingly.

In [1]:
# Run once per environment (remove the leading '#' the first time)
# %pip install -q ultralytics torch torchvision

In [2]:
import csv, json, time, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from ultralytics import RTDETR
import ultralytics

warnings.filterwarnings("ignore")

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__)
print("device      :", torch.cuda.get_device_name(0) if CUDA else "CPU (training will be very slow)")

ultralytics : 8.4.120
torch       : 2.13.0+cpu
device      : CPU (training will be very slow)


### 2. Load the prepared dataset

In [3]:
def find_work_dir(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "dataset" / "dataset_info.json").is_file():
            return c
        if (c / "Student3-Defect Detection" / "dataset" / "dataset_info.json").is_file():
            return c / "Student3-Defect Detection"
    raise FileNotFoundError("dataset_info.json not found - run 00_data_preparation.ipynb first.")

WORK_DIR    = find_work_dir(Path.cwd())
INFO        = json.loads((WORK_DIR / "dataset" / "dataset_info.json").read_text())
CLASS_NAMES = INFO["class_names"]
DATA_YAML   = Path(INFO["paths"]["yolo_yaml"])
RESULTS_DIR = WORK_DIR / "results"; RESULTS_DIR.mkdir(exist_ok=True)
RUNS_DIR    = WORK_DIR / "runs";    RUNS_DIR.mkdir(exist_ok=True)

assert DATA_YAML.is_file(), f"Missing {DATA_YAML} - re-run notebook 00."
print("Work dir :", WORK_DIR)
print("Split    :", INFO["counts"], "| boxes:", INFO["boxes"])
print("Classes  :", CLASS_NAMES)

Work dir : C:\Users\User\Image-Processing\Student3-Defect Detection
Split    : {'train': 965, 'val': 276, 'test': 144} | boxes: {'train': 4097, 'val': 1196, 'test': 608}
Classes  : ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']


### 3. Training configuration

`rtdetr-l` is the smallest pretrained RT-DETR that Ultralytics distributes. The transformer decoder
needs more epochs than a CNN detector to converge on a small dataset, but it also carries far more
COCO-pretrained knowledge, so 150 epochs with early stopping is a reasonable budget.

`imgsz` is 1280 — Student 2's `ALIGN_TARGET`, i.e. exactly what the integrated pipeline hands the
detector. RT-DETR's attention cost grows quickly with resolution, so the batch size is small. **If
a CUDA out-of-memory error occurs, halve `BATCH` first**; only drop `IMGSZ` as a last resort,
because that reintroduces the small-object problem this rebuild was meant to fix.

In [4]:
MODEL_NAME = "rtdetr-l"     # rtdetr-l | rtdetr-x
EPOCHS     = 150 if CUDA else 10
IMGSZ      = INFO["image_size"]      # 1280 - Student 2's ALIGN_TARGET
BATCH      = 4 if CUDA else 1        # halve on out-of-memory
PATIENCE   = 40
SEED       = INFO["random_seed"]
RUN_NAME   = f"{MODEL_NAME}_pcb"

# Same augmentation policy as YOLOv10 so the two are compared fairly. RT-DETR
# in Ultralytics ignores a few of these internally, which is expected.
AUG = dict(
    degrees=8.0, translate=0.10, scale=0.40,
    shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.5,
    hsv_h=0.010, hsv_s=0.40, hsv_v=0.30,
    mosaic=1.0, close_mosaic=15, mixup=0.0,
)

print(f"{MODEL_NAME} | epochs={EPOCHS} | imgsz={IMGSZ} | batch={BATCH} | device={DEVICE}")

rtdetr-l | epochs=10 | imgsz=1280 | batch=1 | device=cpu


### 4. Load the pretrained model

RT-DETR is fine-tuned from COCO weights. Training a Transformer detector from scratch on 693 images
would not converge to anything usable, so the fallback path below is only a last resort.

In [5]:
try:
    model = RTDETR(f"{MODEL_NAME}.pt")
    pretrained = True
except Exception as exc:
    print("Could not fetch pretrained weights:", exc)
    model = RTDETR(f"{MODEL_NAME}.yaml")
    pretrained = False

n_params = sum(p.numel() for p in model.model.parameters())
print(f"Pretrained: {pretrained} | parameters: {n_params/1e6:.2f} M")

Pretrained: True | parameters: 32.97 M


### 5. Train

Ultralytics uses the RT-DETR training recipe (AdamW, Hungarian set-matching loss, no mosaic in the
final epochs). Note that RT-DETR does **not** use anchors or NMS at any point — the decoder emits a
fixed set of queries and each object is matched to exactly one of them.

In [ ]:
t0 = time.perf_counter()
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    seed=SEED,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=pretrained,
    cos_lr=True,
    val=True,
    plots=True,
    verbose=True,
    **AUG,
)
train_time = time.perf_counter() - t0

RUN_DIR = Path(train_results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print(f"\nTraining finished in {train_time/60:.1f} min")
print("Best weights:", BEST_PT)

New https://pypi.org/project/ultralytics/8.4.135 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.120  Python-3.12.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13500HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\User\Image-Processing\Student3-Defect Detection\dataset\yolo\data.yaml, degrees=8.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.4, hsv_v=0.3, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=

### 6. Training curves

In [ ]:
rows = list(csv.DictReader((RUN_DIR / "results.csv").open()))
hist = {k.strip(): [float(r[k]) for r in rows] for k in rows[0]}

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for key in [k for k in hist if k.startswith("train/") and "loss" in k]:
    ax[0].plot(hist["epoch"], hist[key], label=key.split("/")[-1])
ax[0].set_title("Training losses (set-matching)"); ax[0].set_xlabel("epoch"); ax[0].legend()

for key in [k for k in hist if k.startswith("val/") and "loss" in k]:
    ax[1].plot(hist["epoch"], hist[key], label=key.split("/")[-1])
ax[1].set_title("Validation losses"); ax[1].set_xlabel("epoch"); ax[1].legend()

for key, lab in [("metrics/mAP50(B)", "mAP@0.5"), ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    if key in hist:
        ax[2].plot(hist["epoch"], hist[key], label=lab)
ax[2].set_title("Validation mAP"); ax[2].set_xlabel("epoch"); ax[2].legend()
plt.tight_layout(); plt.show()

### 7. Evaluation on the held-out test split

In [ ]:
best = RTDETR(str(BEST_PT))
metrics = best.val(
    data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, project=str(RUNS_DIR), name=f"{RUN_NAME}_test",
    exist_ok=True, plots=True, verbose=False,
)

box = metrics.box
summary = {
    "precision": float(box.mp),
    "recall":    float(box.mr),
    "mAP50":     float(box.map50),
    "mAP75":     float(box.map75),
    "mAP50_95":  float(box.map),
}
summary["f1"] = (2 * summary["precision"] * summary["recall"] /
                 max(summary["precision"] + summary["recall"], 1e-9))

print(f"{'metric':<14}{'value':>9}")
for k, v in summary.items():
    print(f"{k:<14}{v:>9.4f}")

### 8. Per-class results

Comparing this table with the YOLOv10 one shows where global attention helps: RT-DETR usually gains
most on the classes that are easily confused with the repetitive background trace pattern
(`spurious_copper`, `spur`), because the encoder sees the whole board rather than a local
receptive field.

In [ ]:
per_class = {}
for idx, cls_idx in enumerate(box.ap_class_index):
    name = CLASS_NAMES[int(cls_idx)]
    per_class[name] = {
        "precision": float(box.p[idx]),
        "recall":    float(box.r[idx]),
        "mAP50":     float(box.ap50[idx]),
        "mAP50_95":  float(box.ap[idx].mean()) if box.ap[idx].ndim else float(box.ap[idx]),
    }

print(f"{'class':<18}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>10}")
for n, m in per_class.items():
    print(f"{n:<18}{m['precision']:>8.3f}{m['recall']:>8.3f}{m['mAP50']:>9.3f}{m['mAP50_95']:>10.3f}")

names = list(per_class)
x = np.arange(len(names))
plt.figure(figsize=(9, 4))
plt.bar(x - 0.2, [per_class[n]["mAP50"] for n in names], 0.4, label="mAP@0.5")
plt.bar(x + 0.2, [per_class[n]["mAP50_95"] for n in names], 0.4, label="mAP@0.5:0.95")
plt.xticks(x, names, rotation=30, ha="right"); plt.ylim(0, 1)
plt.title(f"{MODEL_NAME} - per-class AP on the test split"); plt.legend(); plt.tight_layout(); plt.show()

### 9. Inference speed

Identical protocol to notebooks 01 and 02. The number to watch is the **standard deviation**: for
an NMS-free model it stays small regardless of how many defects a board carries, which is the
"constant-latency envelope" claimed in the literature review.

In [ ]:
test_images = sorted((Path(INFO["paths"]["yolo_root"]) / "images" / "test").glob("*.jpg"))
for p in test_images[:10]:
    best.predict(str(p), imgsz=IMGSZ, device=DEVICE, verbose=False)

latencies = []
for p in test_images:
    t = time.perf_counter()
    best.predict(str(p), imgsz=IMGSZ, device=DEVICE, verbose=False)
    latencies.append((time.perf_counter() - t) * 1000)

lat_mean, lat_std = float(np.mean(latencies)), float(np.std(latencies))
fps = 1000.0 / lat_mean
print(f"Images timed      : {len(test_images)}")
print(f"Latency per image : {lat_mean:.2f} +/- {lat_std:.2f} ms")
print(f"Throughput        : {fps:.1f} FPS  (NMS-free end-to-end)")
print(f"Ultralytics profile: {metrics.speed}")

### 10. Qualitative results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, p in zip(axes.ravel(), test_images[:6]):
    r = best.predict(str(p), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)[0]
    ax.imshow(Image.open(p))
    for b, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                        r.boxes.cls.cpu().numpy().astype(int),
                        r.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = b
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                   edgecolor="red", linewidth=1.8))
        ax.text(x1, max(y1 - 4, 8), f"{CLASS_NAMES[c]} {cf:.2f}", color="yellow", fontsize=7,
                bbox=dict(facecolor="black", alpha=.6, pad=1))
    ax.set_title(p.stem, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

### 11. Confusion matrix

In [ ]:
cm_path = RUN_DIR.parent / f"{RUN_NAME}_test" / "confusion_matrix_normalized.png"
if cm_path.exists():
    plt.figure(figsize=(8, 7)); plt.imshow(Image.open(cm_path)); plt.axis("off"); plt.show()
else:
    print("Confusion matrix image not found at", cm_path)

### 12. Save the results

In [ ]:
payload = {
    "model": "RT-DETR",
    "variant": MODEL_NAME,
    "framework": f"ultralytics {ultralytics.__version__}",
    "type": "end-to-end transformer (NMS-free)",
    "pretrained": pretrained,
    "params_M": round(n_params / 1e6, 2),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": "GPU" if CUDA else "CPU",
    "train_time_min": round(train_time / 60, 2),
    "test": summary,
    "per_class": per_class,
    "latency_ms": round(lat_mean, 2),
    "latency_std_ms": round(lat_std, 2),
    "fps": round(fps, 1),
    "weights": str(BEST_PT),
}
out = RESULTS_DIR / "rtdetr.json"
out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved", out)
print(json.dumps({k: v for k, v in payload.items() if k != "per_class"}, indent=2))